# Creating Training Data

For the binary image classification task, you need to know what is class A and what is class B. For our project, we needed many examples of maps and non-maps, which had to be gathered and classified by hand. If your classification model involves finetuning a model (e.g. logistic regression, a convolutional neural network, or an encoder model like BERT), then you need to have enough examples in your training data to both train and test your model. If your classificaiton model uses zero-shot classification (e.g. Gemini), then you really only need enough examples to test it, which reduces how much manual annotating you need to do.

We gathered 1000 page images which included maps and non-maps. Since maps are relatively rare, and can come in a wide variety of styles, we wanted to have a lot of these maps, so we created an unbalanced dataset of 730 maps and 270 non-maps. Ideally, you want the ratio to be closer to the actual ratio in your corpus, but maps are so rare that we would have needed millions of text pages just to get 1000 maps with the same ratio (remember, only 0.008% of pages contaiend maps). So in this needle-in-a-haystack situation, the unbalanced training data is acceptable.

Let's assume that you have now gathered some training data, which is all in the same directory and you have a csv noting which images are which class:

In [ ]:
image_folder = 'images' # there should be ~1000+ images here
sorted_files = 'Classifier_images.csv'
'''this csv file has several columns. The important ones are:
Document: htid id of the novel image was pulled from (or any unique id in your case)
Source: title of the novel image was pulled from (not necessary)
ID: unique page id. in this case htid + page number (this should be unique for every image)
Class: integer of classes. We used 0-2, but for a binary task, it should be 0-1
Class_ID: string of class. USeful while your tagging, but not necessary for our code

(we added a sample of Classifier_images.csv in case you need to reference it)'''

(Note: you should make a copy of this image folder, before proceeding, we will be manipulating its contents!)

In [ ]:
import pandas as pd
import cv2 
import random
import os
import math
import shutil
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#change these based on your own files
sorted_files = 'Classifier_images.csv'
path = 'your/local/path'

In [ ]:
to_load = os.path.join(path, sorted_files)
images = pd.read_csv(to_load, dtype={'Document': 'str', 'Source': 'str', 'ID':'str', 'Class':'int', 'Class_ID':'str'},
                     encoding='utf-8', sep=',', quoting=3)
img_only = images[images['Class'] == 2]
non_imgs = images[images['Class'] != 2] # during annotation we separated text and illustrations in case this was helpful, but we collapse these categories into a single non-map class for actual analysis.
img_only = img_only.reset_index()
non_imgs = non_imgs.reset_index()
img_only.info()

print("maps:", len(img_only))
print("text & images:", len(non_imgs))

#empty train and test folders
train_c1 = os.path.join(path, 'images/train/class1')
train_c2 = os.path.join(path, 'images/train/class2')
test_c1 = os.path.join(path, 'images/test/class1')
test_c2 = os.path.join(path, 'images/test/class2')

files = os.listdir(train_c1)
for f in files:
    f = os.path.join(train_c1, f)
    os.remove(f)
files = os.listdir(train_c2)
for f in files:
    f = os.path.join(train_c2, f)
    os.remove(f)
files = os.listdir(test_c1)
for f in files:
    f = os.path.join(test_c1, f)
    os.remove(f)
files = os.listdir(test_c2)
for f in files:
    f = os.path.join(test_c2, f)
    os.remove(f)
    
#initialize training and testing (randomly select 80/20 split) for maps
perc_train = 0.8
random.seed(42)

img_only['Order'] = 0.0
for i in img_only.index:
    img_only.loc[i, 'Order'] = random.random()
img_only = img_only.sort_values(by=['Order'])
img_only = img_only.reset_index()

print(math.floor(perc_train*len(img_only)))
#sort images into appropriate folders
img_only['TT'] = ''
for i in img_only.index:
    #print(i)
    if i <= math.floor(perc_train*len(img_only)):
        img_only.loc[i, 'TT'] = 'train'
        shutil.copy2(os.path.join(path, 'images/') + img_only.loc[i, 'ID'] + '.jpg', train_c1)
    else:
        img_only.loc[i, 'TT'] = 'test'
        shutil.copy2(os.path.join(path, 'images/') + img_only.loc[i, 'ID'] + '.jpg', test_c1)
print(img_only.head())
print(img_only.tail())

#initialize training and testing (randomly select 80/20 split) for non maps
perc_train = 0.8

non_imgs['Order'] = 0.0
for i in non_imgs.index:
    non_imgs.loc[i, 'Order'] = random.random()
non_imgs = non_imgs.sort_values(by=['Order'])
non_imgs = non_imgs.reset_index()

print(math.floor(perc_train*len(non_imgs)))
#sort images into appropriate folders
non_imgs['TT'] = ''
for i in non_imgs.index:
    #print(i)
    if i <= math.floor(perc_train*len(non_imgs)):
        non_imgs.loc[i, 'TT'] = 'train'
        shutil.copy2(os.path.join(path, 'images/') + non_imgs.loc[i, 'ID'] + '.jpg', train_c2)
    else:
        non_imgs.loc[i, 'TT'] = 'test'
        shutil.copy2(os.path.join(path, 'images/') + non_imgs.loc[i, 'ID'] + '.jpg', test_c2)
non_imgs.head()

Now you should have all of your images sorted into folders for each class with a 80/20 train/test split.

For image classification tasks, it is often beneficial to manipulate your images to supplement your dataset. This will make the classifier more robust to noise/rotations and it will also extend your training data.

In [ ]:
# define some functions to manipulate images randomly

image_size = 512

def rand_rotation(image): #randomly rotates image 90, 180, or 270 degrees
    num_rot = random.randint(0, 3)
    for i in range(num_rot):
        image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    return image

def rand_flip(image): #randomly flips either horizontally or vertically
    dir_flip = random.randint(-1, 1)
    image = cv2.flip(image, dir_flip)
    return image

def rand_patch(image, size): #take a random patch from the image.
    #0 < size < 1
    #check size of image
    height, width = image.shape[:2]
    if height != width:
        print('image not square')
        return KeyError
    #determine patch size
    h_start = random.randint(0, height-round(height*size))
    w_start = random.randint(0, width-round(width*size))
    image = image[h_start:h_start+round(height*size), w_start:w_start+round(width*size)]
    #resize image back to original size
    image = cv2.resize(image, (height, width), interpolation=cv2.INTER_CUBIC)
    return image

def rand_patch_size(image, min_size, max_size): #define min/max sizes for random patches
    min_size = round(min_size*100)
    max_size = round(max_size*100)
    size = random.randint(min_size, max_size)
    size = round(size/100, 2)
    image = rand_patch(image, size)
    return image, size

def rand_noise(image, min_noise, max_noise): #add some gaussian noise to the image
    #values between 500 and 5000
    mean = 0
    var = random.randint(min_noise, max_noise)
    sigma = var ** 0.5
    gaussian = np.random.normal(mean, sigma, image.shape)
    # Add the noise to the image
    noisy_image = image + gaussian
    # Clip values to ensure they are within the valid range [0, 255]
    noisy_image = np.clip(noisy_image, 0, 255)
    # Convert the image back to uint8
    noisy_image = noisy_image.astype(np.uint8)
    return noisy_image, var

In [ ]:
non_imgs['patch'] = '0'
non_imgs['flip'] = '0'
non_imgs['rotate'] = '0'
img_only['patch'] = '0'
img_only['flip'] = '0'
img_only['rotate'] = '0'

random.seed()
for j in range(2): # create 2 manipulated images alongside each original
    for i in non_imgs.index:
        print(i)
        if non_imgs.loc[i, 'TT'] == 'train':
            img = os.path.join(path, 'images/train/class2/') + non_imgs.loc[i, 'ID'] + '.jpg'
            image = cv2.imread(img, cv2.IMREAD_GRAYSCALE)
            image = cv2.resize(image, (image_size, image_size))
            #print(image)
            #generate possible manipulations
            var1 = 1            
            if var1 == 1:
                image, size = rand_patch_size(image, 0.3, 0.7)
                size = round(size * 10)
                non_imgs.loc[i, 'patch'] = 'p' + str(size)
                print('patched')
            var1 = 1
            if var1 == 1:
                image, noise = rand_noise(image, 500, 4000)
                non_imgs.loc[i, 'gnoise'] = 'g' + str(noise)
                print('gaussian noise added:', noise)
            var1 = random.randint(0,1)
            var1 = random.randint(0,1)
            if var1 == 1:
                image = rand_flip(image)
                non_imgs.loc[i, 'flip'] = 'f'
                print('flipped')
            var1 = random.randint(0,1)
            if var1 == 1:
                image = rand_rotation(image)
                non_imgs.loc[i, 'rotate'] = 'r'
                print('rotated')

            new_img_name = os.path.join(path, 'images/train/class2/') + non_imgs.loc[i, 'ID'] + '_' + str(j) + '_' + non_imgs.loc[i, 'patch'] + non_imgs.loc[i, 'gnoise'] + non_imgs.loc[i, 'flip'] + non_imgs.loc[i, 'rotate'] + '_comb.jpg'
            #new_img_name = './images/train/{0}/{1}{2}.jpg'.format('class2', image, 'comb')
            #print(new_img_name)
            cv2.imwrite(new_img_name, image)

    for i in img_only.index:
        #print(i)
        if img_only.loc[i, 'TT'] == 'train':
            img = os.path.join(path, 'images/train/class1/') + img_only.loc[i, 'ID'] + '.jpg'
            image = cv2.imread(img, cv2.IMREAD_GRAYSCALE)
            image = cv2.resize(image, (image_size, image_size))
            #print(image)
            #generate possible manipulations
            var1 = 1            
            if var1 == 1:
                image, size = rand_patch_size(image, 0.3, 0.7)
                size = round(size * 10)
                img_only.loc[i, 'patch'] = 'p' + str(size)
                print('patched')
                var1 = 1
            if var1 == 1:
                image, noise = rand_noise(image, 500, 4000)
                img_only.loc[i, 'gnoise'] = 'g' + str(noise)
                print('gaussian noise added:', noise)
            var1 = random.randint(0,1)
            if var1 == 1:
                image = rand_flip(image)
                img_only.loc[i, 'flip'] = 'f'
                print('flipped')
            var1 = random.randint(0,1)
            if var1 == 1:
                image = rand_rotation(image)
                img_only.loc[i, 'rotate'] = 'r'
                print('rotated')

            new_img_name = os.path.join(path, 'images/train/class1/') + img_only.loc[i, 'ID'] + '_' + str(j) + '_' + img_only.loc[i, 'patch'] + img_only.loc[i, 'gnoise'] + img_only.loc[i, 'flip'] + img_only.loc[i, 'rotate'] + '_comb.jpg'
            #new_img_name = './images/train/{0}/{1}{2}.jpg'.format('class1', image, 'comb')
            #print(new_img_name)
            cv2.imwrite(new_img_name, image)



Now these directories can be used for finetuning a CNN model!